# 🛠️ Notebook 2: Airline Management — Implementation

We implement the clean design from Notebook 1, one concept at a time:

1. Core types (Passenger, Seat, Aircraft, Flight).
2. Booking + pricing table.
3. Cancellation with a refund policy.
4. Search across many flights.
5. Crew assignment.
6. Concurrency: a realistic race condition and how to fix it.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/airline-management
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Core types

- `Passenger` and `Seat` are **immutable value objects** (`frozen=True`): two seats with the same number are "the same seat."
- `SeatClass` is an **enum** — fixed set of choices, no typos.
- `PRICING` is a tiny **lookup table**. Adding Premium Economy = one new line. No subclassing.


In [1]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from itertools import count
from threading import Lock

class SeatClass(Enum):
    ECONOMY = "ECON"
    BUSINESS = "BIZ"
    FIRST = "FIRST"

# Data-shaped variation → lookup table, not subclasses.
PRICING = {
    SeatClass.ECONOMY:  200,
    SeatClass.BUSINESS: 700,
    SeatClass.FIRST:   1800,
}

@dataclass(frozen=True)
class Passenger:
    id: int
    name: str
    passport: str

@dataclass(frozen=True)
class Seat:
    number: str               # e.g. "12A"
    seat_class: SeatClass

@dataclass
class Aircraft:
    model: str                # e.g. "A320"
    seats: list[Seat]

# Quick sanity check
demo_seats = [Seat("1A", SeatClass.FIRST), Seat("5A", SeatClass.ECONOMY)]
print(Aircraft("A320", demo_seats))


Aircraft(model='A320', seats=[Seat(number='1A', seat_class=<SeatClass.FIRST: 'FIRST'>), Seat(number='5A', seat_class=<SeatClass.ECONOMY: 'ECON'>)])


## 2. `Flight` + `Booking`

Notice how `Flight` owns only **availability** — the seat *map* lives on `Aircraft`. That way the same A320 can fly UA100 today and UA250 tomorrow without duplicating seats.

`Booking` is its own object. It has a `status` so we can cancel it without losing the audit trail.


In [2]:
_bids = count(1)   # simple booking id generator

class BookingStatus(Enum):
    CONFIRMED = "CONFIRMED"
    CANCELLED = "CANCELLED"

@dataclass
class Booking:
    passenger: Passenger
    flight: "Flight"
    seat: Seat
    price: float
    status: BookingStatus = BookingStatus.CONFIRMED
    id: int = field(default_factory=lambda: next(_bids))

    def ticket(self) -> str:
        return (f"TKT#{self.id}  {self.flight.number}  "
                f"{self.flight.origin}→{self.flight.destination}  "
                f"seat {self.seat.number} ({self.seat.seat_class.value})  "
                f"${self.price}  [{self.status.value}]")

@dataclass
class Flight:
    number: str
    origin: str
    destination: str
    departs: datetime
    aircraft: Aircraft
    _available: dict[str, bool] = field(init=False)
    _lock: Lock = field(init=False, repr=False)

    def __post_init__(self):
        self._available = {s.number: True for s in self.aircraft.seats}
        self._lock = Lock()     # protects concurrent bookings (see section 6)

    def available_seats(self, seat_class: SeatClass | None = None) -> list[Seat]:
        return [s for s in self.aircraft.seats
                if self._available[s.number]
                and (seat_class is None or s.seat_class == seat_class)]

    def book(self, passenger: Passenger, seat: Seat) -> Booking:
        with self._lock:
            if not self._available.get(seat.number, False):
                raise ValueError(f"seat {seat.number} not available")
            self._available[seat.number] = False
        return Booking(passenger, self, seat, PRICING[seat.seat_class])

    def cancel(self, booking: Booking, now: datetime | None = None) -> float:
        '''Cancel a booking and return the refunded amount.

        Refund policy (simple, realistic):
          >= 7 days before departure → 100% refund
          >= 1 day                   → 50% refund
          otherwise                  → 0% refund
        '''
        if booking.flight is not self:
            raise ValueError("booking is not on this flight")
        if booking.status is BookingStatus.CANCELLED:
            raise ValueError("already cancelled")
        now = now or datetime.now()
        lead = self.departs - now
        if lead >= timedelta(days=7):
            refund_pct = 1.0
        elif lead >= timedelta(days=1):
            refund_pct = 0.5
        else:
            refund_pct = 0.0
        with self._lock:
            self._available[booking.seat.number] = True
        booking.status = BookingStatus.CANCELLED
        return round(booking.price * refund_pct, 2)


### Try it: book, ticket, double-book, cancel

In [3]:
# Tiny A320-ish seat map: 4 economy + 2 business + 1 first
seats = (
    [Seat(f"{r}{c}", SeatClass.ECONOMY)  for r in (5, 6) for c in "AB"]
    + [Seat(f"{r}{c}", SeatClass.BUSINESS) for r in (2,)  for c in "AB"]
    + [Seat("1A", SeatClass.FIRST)]
)
plane = Aircraft("A320", seats)

fl = Flight("UA100", "SFO", "JFK",
            datetime(2025, 6, 1, 9, 0), plane)

alice = Passenger(1, "Alice", "P111")
bob   = Passenger(2, "Bob",   "P222")

print("Business seats available:", fl.available_seats(SeatClass.BUSINESS))

b1 = fl.book(alice, fl.available_seats(SeatClass.BUSINESS)[0])
b2 = fl.book(bob,   fl.available_seats(SeatClass.ECONOMY)[0])
print(b1.ticket())
print(b2.ticket())

# Double-booking the same seat must fail
try:
    fl.book(bob, b1.seat)
except ValueError as e:
    print("expected error:", e)

# Cancel Alice's booking 10 days before departure → full refund
refund = fl.cancel(b1, now=fl.departs - timedelta(days=10))
print(f"Alice refund: ${refund}  |  seat free again? "
      f"{b1.seat in fl.available_seats()}")


Business seats available: [Seat(number='2A', seat_class=<SeatClass.BUSINESS: 'BIZ'>), Seat(number='2B', seat_class=<SeatClass.BUSINESS: 'BIZ'>)]
TKT#1  UA100  SFO→JFK  seat 2A (BIZ)  $700  [CONFIRMED]
TKT#2  UA100  SFO→JFK  seat 5A (ECON)  $200  [CONFIRMED]
expected error: seat 2A not available
Alice refund: $700.0  |  seat free again? True


## 3. Searching flights

`FlightSearch` is a separate service — it *queries* flights; it doesn't *own* them. Putting search inside `Flight` would violate single responsibility and make it hard to add filters (airline, stops, price…).


In [4]:
@dataclass
class FlightSearch:
    flights: list[Flight]

    def search(self, origin: str, destination: str, date) -> list[Flight]:
        return [f for f in self.flights
                if f.origin == origin
                and f.destination == destination
                and f.departs.date() == date]

# Build a small schedule: 3 flights on different days / routes
plane2 = Aircraft("A320", [Seat("5A", SeatClass.ECONOMY)])
schedule = [
    fl,
    Flight("UA200", "SFO", "JFK", datetime(2025, 6, 1, 18, 0), plane2),
    Flight("UA300", "SFO", "LAX", datetime(2025, 6, 1, 8, 0),  plane2),
]
search = FlightSearch(schedule)
results = search.search("SFO", "JFK", datetime(2025, 6, 1).date())
print("Found", len(results), "SFO→JFK flights on 2025-06-01:")
for f in results:
    print(" ", f.number, f.departs.strftime("%H:%M"))


Found 2 SFO→JFK flights on 2025-06-01:
  UA100 09:00
  UA200 18:00


## 4. Crew

Pilots and attendants are *not* passengers — different role, different lifecycle. Keep them separate.


In [5]:
class CrewRole(Enum):
    PILOT = "PILOT"
    ATTENDANT = "ATTENDANT"

@dataclass(frozen=True)
class CrewMember:
    id: int
    name: str
    role: CrewRole

# Attach crew to a flight without touching Flight's booking logic.
fl.crew = [
    CrewMember(101, "Capt. Smith", CrewRole.PILOT),
    CrewMember(102, "F.O. Jones",  CrewRole.PILOT),
    CrewMember(201, "Maya",        CrewRole.ATTENDANT),
]
print(f"{fl.number} crew:")
for m in fl.crew:
    print(" -", m.role.value, m.name)


UA100 crew:
 - PILOT Capt. Smith
 - PILOT F.O. Jones
 - ATTENDANT Maya


## 5. Concurrency — the realistic bug

Two customers on two browser tabs click "Book seat 2A" at almost the same moment.

Without a lock, *both* checks see the seat as free and *both* succeed. That's a real production bug in booking systems.

Our `Flight.book` already uses a `Lock` — let's prove it matters by forcing the race with many threads.


In [6]:
from threading import Thread

# Fresh flight with ONE business seat — the contested one.
tiny = Aircraft("A320", [Seat("2A", SeatClass.BUSINESS)])
hot_flight = Flight("UA999", "SFO", "JFK", datetime(2025, 6, 1, 9, 0), tiny)

winners, losers = [], []

def try_book(name):
    p = Passenger(hash(name) & 0xFFFF, name, "PX")
    try:
        b = hot_flight.book(p, Seat("2A", SeatClass.BUSINESS))
        winners.append(b)
    except ValueError:
        losers.append(name)

threads = [Thread(target=try_book, args=(f"user{i}",)) for i in range(50)]
for t in threads: t.start()
for t in threads: t.join()

print(f"winners: {len(winners)}  losers: {len(losers)}")
assert len(winners) == 1, "lock is broken — two people got the same seat!"
print("✅ Exactly one booking succeeded, as expected.")


winners: 1  losers: 49
✅ Exactly one booking succeeded, as expected.


### What would be *wrong* without the lock?

If we replaced `Flight.book` with:

```python
def book(self, passenger, seat):
    if not self._available[seat.number]:   # check
        raise ValueError("not available")
    self._available[seat.number] = False   # mutate
    return Booking(...)
```

Between *check* and *mutate*, another thread can sneak in — both threads see `True`, both set `False`, both return bookings. This is called a **check-then-act race**. Always guard check+mutate pairs with a lock (or use an atomic operation / a DB transaction in production).


## 🎯 Your turn — small, high-value extensions

- **Overbooking**: allow up to 5% more bookings than seats; assign overflow at check-in.
- **Seat holds**: `hold(seat, ttl=10min)` prevents others from booking while the passenger enters payment. Release on expiry.
- **Loyalty discount**: wrap `PRICING` in a function `price_for(passenger, seat_class)` — no new classes needed.
- **Multi-leg itineraries**: a `Trip` that owns many `Booking`s on connecting `Flight`s, with a combined price.
- **Persistence**: serialize flights/bookings with `dataclasses.asdict`, reload on restart.


## 🧠 Takeaways

1. **Start from requirements** — not from classes. Classes fall out of responsibilities.
2. **Subclass for behavior, not for data.** Seat classes vary in *price*, not in *what a seat does* → use an enum + lookup.
3. **Separate concerns**: seat map (`Aircraft`) vs availability (`Flight`) vs transaction (`Booking`) vs query (`FlightSearch`).
4. **Booking is a first-class object** — with a status — so cancellation and audits are trivial.
5. **Concurrency matters**: check-then-act without a lock is a bug. A `Lock` around the critical section is enough for a single process; in production use your DB transaction.
